# Práctica 9: Clasificación: softmax, entropía cruzada y generalización

**Fundamentos matemáticos del aprendizaje profundo** · Departamento de Matemática, FCEN-UBA

Material de referencia: capítulos 10 de las notas del curso. Convenciones: $y=\varphi(w\cdot x+b)$, $\sigma$ es la logística, $\eta$ la tasa de aprendizaje, $w_{ij}$ conecta la entrada $i$ con la salida $j$.

Cada práctica tiene una **Parte A** (ejercicios teóricos, para resolver en papel) y una **Parte B** (computacional). Las celdas marcadas `# TODO` son para completar.

## Parte A. Ejercicios teóricos

**A1.** Sea $z=\mathrm{softmax}(s)$, $z_k=e^{s_k}/\sum_je^{s_j}$. Mostrar $\partial z_k/\partial s_j=z_k(\delta_{kj}-z_j)$ y que con $C=-\sum_ky_k\ln z_k$ ($y$ one-hot) se obtiene $\partial C/\partial s=z-y$.

**A2.** Mostrar que softmax es invariante por traslaciones $s\mapsto s+c\mathbf 1$ y explicar el truco de restar $\max_ks_k$ para estabilidad numérica.

**A3.** Mostrar que para $K=2$ softmax se reduce a la logística aplicada a $s_1-s_2$. Interpretar.

**A4.** Un clasificador lineal con $K$ clases decide $\arg\max_k(w_k\cdot x+b_k)$. Mostrar que las regiones de decisión son convexas.

**A5.** Sea $S$ la relación sobre clasificadores definida en clase. Demostrar que es transitiva y decidir si es de equivalencia.

**A6.** (Generalización) Sea $\mathcal H$ una clase finita y $\ell\in[0,1]$. Mostrar con Hoeffding que con probabilidad $\ge1-\delta$, $\sup_{h\in\mathcal H}|R(h)-\hat R_N(h)|\le\sqrt{\frac{\ln(2|\mathcal H|/\delta)}{2N}}$.

## Parte B. Computacional

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch
from torch import nn
torch.manual_seed(0); rng=np.random.default_rng(0)
plt.rcParams['figure.figsize']=(6,4)

### B1. Softmax estable y gradiente
Implementar `softmax` y `cross_entropy` en NumPy (con estabilización) y verificar $\partial C/\partial s=z-y$ por diferencias finitas.

In [ ]:
def softmax(s):
    s=s-s.max(axis=-1,keepdims=True)
    e=np.exp(s); return e/e.sum(axis=-1,keepdims=True)
def ce(s,y): return -np.sum(y*np.log(softmax(s)))
s=rng.standard_normal(4); y=np.eye(4)[2]
g=softmax(s)-y
h=1e-6
gnum=np.array([(ce(s+h*e,y)-ce(s-h*e,y))/(2*h) for e in np.eye(4)])
print(np.abs(g-gnum).max())
print(softmax(np.array([1000.,1001.,1002.])))   # no debe dar nan

### B2. Clasificador softmax sobre datos 2D (3 clases)
Generar 3 nubes gaussianas, entrenar una regresión softmax lineal con PyTorch y graficar las regiones de decisión (A4: regiones convexas).

In [ ]:
K,n=3,100
centros=np.array([[0,2],[-2,-1],[2,-1]])
X=np.vstack([rng.normal(c,0.8,(n,2)) for c in centros]).astype(np.float32)
y=np.repeat(np.arange(K),n)
Xt,yt=torch.tensor(X),torch.tensor(y)
modelo=nn.Linear(2,K)
opt=torch.optim.Adam(modelo.parameters(),lr=0.05)
for ep in range(300):
    loss=nn.functional.cross_entropy(modelo(Xt),yt)
    opt.zero_grad(); loss.backward(); opt.step()
print(loss.item())
# TODO: regiones de decision con meshgrid + contourf

### B3. Datos no linealmente separables: dos medias lunas
Usar `sklearn.datasets.make_moons`. Comparar un modelo lineal con un MLP `2-16-16-2`. Graficar regiones de decisión.

In [ ]:
# TODO

### B4. Sobreajuste y conjuntos de validación
Con `make_moons(noise=0.35)` y pocos datos de entrenamiento (30), entrenar un MLP grande hasta interpolar. Graficar las curvas de costo de entrenamiento y validación. Probar regularización $L^2$ (`weight_decay`) y *early stopping*.

In [ ]:
# TODO

### B5. Dígitos (clasificación real)
Cargar `sklearn.datasets.load_digits` ($8\times8$), separar entrenamiento/test (80/20) y entrenar un MLP `64-64-10`. Reportar la exactitud y graficar la matriz de confusión y algunos errores. (Para MNIST completo usar `torchvision.datasets.MNIST` en Colab.)

In [ ]:
# TODO

### B6. Tasa de generalización
Entrenar el modelo de B5 con $N=50,100,200,500,1000$ datos y graficar el error de test vs. $N$ (log-log). Comparar cualitativamente con la cota de A6.

In [ ]:
# TODO